# Thematic Analysis — Keyword Grouping Logic

This notebook documents how themes were defined and assigned.
Theme definitions were derived from actual TF-IDF keyword exploration
on the dataset — not from generic assumptions.

## Step 1: TF-IDF Keyword Discovery

Before defining any themes, TF-IDF was run separately on:
- All reviews per bank
- Negative reviews only (rating <= 2)
- Positive reviews only (rating >= 4)

This revealed what users actually write about for each bank.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('../data/raw/reviews_clean.csv')

for bank in df['bank'].unique():
    print(f"\n{'='*55}")
    print(f"TOP NEGATIVE KEYWORDS: {bank}")
    print('='*55)
    
    negative = df[
        (df['bank'] == bank) & (df['rating'] <= 2)
    ]['review'].tolist()
    
    if len(negative) < 3:
        continue
    
    vec = TfidfVectorizer(
        max_features=20,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2
    )
    matrix = vec.fit_transform(negative)
    names = vec.get_feature_names_out()
    scores = matrix.sum(axis=0).A1
    words = sorted(zip(names, scores), key=lambda x: x[1], reverse=True)
    
    for word, score in words[:15]:
        print(f"  {word:<30} {score:.3f}")


TOP NEGATIVE KEYWORDS: Commercial Bank of Ethiopia
  app                            20.549
  update                         9.798
  work                           9.538
  working                        7.316
  transfer                       6.613
  application                    6.551
  service                        4.558
  unable                         4.228
  doesn                          4.186
  bank                           4.165
  fix                            4.142
  version                        3.621
  money                          3.546
  new                            3.347
  transactions                   3.039

TOP NEGATIVE KEYWORDS: Bank of Abyssinia
  app                            51.169
  work                           15.115
  working                        14.515
  worst                          14.174
  bank                           13.855
  fix                            10.351
  banking                        10.134
  doesn                          9.792
 

## Step 2: Keyword Grouping Logic

Based on the TF-IDF output above, keywords were grouped
into themes by the business concept they represent:

| Keywords Discovered | Business Concept | Theme |
|--------------------|-----------------|-------|
| transfer, slow, payment, loading, money | Moving money | Transaction Performance |
| work, working, fix, update, crash, error, unable | App reliability | App Stability |
| login, otp, password, account, access, locked | Getting into app | Account Access |
| easy, nice, good, great, interface, smooth | How app feels | User Experience |
| support, help, response, complaint, agent | Getting help | Customer Service |
| add, need, want, suggest, improve, feature | Missing features | Feature Requests |

## Step 3: Assignment Rule

```python
THEME_KEYWORDS = {
    'Transaction Performance': ['transfer', 'payment', 'slow', ...],
    'App Stability':           ['crash', 'fix', 'update', ...],
    'Account Access':          ['login', 'otp', 'password', ...],
    'User Experience':         ['easy', 'nice', 'good', ...],
    'Customer Service':        ['support', 'help', 'response', ...],
    'Feature Requests':        ['add', 'need', 'want', ...]
}
```

First matching theme wins.
Reviews matching no theme → labeled **General**.

## Step 2: Keyword Grouping Logic

Based on the TF-IDF output above, keywords were grouped
into themes by the business concept they represent:

| Keywords Discovered | Business Concept | Theme |
|--------------------|-----------------|-------|
| transfer, slow, payment, loading, money | Moving money | Transaction Performance |
| work, working, fix, update, crash, error, unable | App reliability | App Stability |
| login, otp, password, account, access, locked | Getting into app | Account Access |
| easy, nice, good, great, interface, smooth | How app feels | User Experience |
| support, help, response, complaint, agent | Getting help | Customer Service |
| add, need, want, suggest, improve, feature | Missing features | Feature Requests |

## Step 3: Assignment Rule

```python
THEME_KEYWORDS = {
    'Transaction Performance': ['transfer', 'payment', 'slow', ...],
    'App Stability':           ['crash', 'fix', 'update', ...],
    'Account Access':          ['login', 'otp', 'password', ...],
    'User Experience':         ['easy', 'nice', 'good', ...],
    'Customer Service':        ['support', 'help', 'response', ...],
    'Feature Requests':        ['add', 'need', 'want', ...]
}
```

First matching theme wins.
Reviews matching no theme → labeled **General**.